In [ ]:
import pandas as pd
import requests
import time

# --- CONFIGURACIÓN ---
API_KEY = "TU_API_KEY_AQUÍ"  # 🔑 Pega aquí tu clave de RAWG
ARCHIVO_ENTRADA = "backloggd_juegos_con_logs.xlsx"
ARCHIVO_SALIDA = "backloggd_juegos_enriquecido.xlsx"
ARCHIVO_NO_ENCONTRADOS = "rawg_no_encontrados.txt"

# --- Leer Excel original ---
df = pd.read_excel(ARCHIVO_ENTRADA)

# --- Añadir columnas si no existen ---
nuevas_columnas = [
    "Nombre RAWG", "Géneros", "Desarrollador",
    "Plataformas", "Fecha de salida", "Metacritic"
]
for col in nuevas_columnas:
    if col not in df.columns:
        df[col] = ""

# --- Función para consultar juego en RAWG ---
def obtener_datos_rawg(nombre):
    try:
        url_busqueda = "https://api.rawg.io/api/games"
        params = {"search": nombre, "key": API_KEY, "page_size": 1}
        res = requests.get(url_busqueda, params=params)
        if res.status_code != 200:
            return None
        data = res.json()
        if not data.get("results"):
            return None

        juego = data["results"][0]
        slug = juego["slug"]

        # Consulta detallada
        url_detalle = f"https://api.rawg.io/api/games/{slug}?key={API_KEY}"
        detalle = requests.get(url_detalle).json()

        return {
            "Nombre RAWG": juego.get("name", ""),
            "Géneros": ", ".join([g["name"] for g in juego.get("genres", [])]),
            "Desarrollador": ", ".join([d["name"] for d in detalle.get("developers", [])]) if "developers" in detalle else "",
            "Plataformas": ", ".join([p["platform"]["name"] for p in juego.get("platforms", [])]),
            "Fecha de salida": juego.get("released", ""),
            "Metacritic": juego.get("metacritic", "")
        }
    except Exception as e:
        print(f"⚠️ Error consultando '{nombre}': {e}")
        return None

# --- Recorrer juegos ---
no_encontrados = []

for index, row in df.iterrows():
    nombre = row["Juego"]
    print(f"🔍 ({index + 1}/{len(df)}) Buscando: {nombre}")

    datos = obtener_datos_rawg(nombre)

    if datos:
        for key, value in datos.items():
            df.at[index, key] = value
    else:
        print(f"❌ No encontrado: {nombre}")
        no_encontrados.append(nombre)

    time.sleep(1.2)  # Esperar entre peticiones

# --- Guardar resultados ---
df.to_excel(ARCHIVO_SALIDA, index=False)
print(f"\n✅ Archivo guardado: {ARCHIVO_SALIDA}")

# --- Guardar no encontrados (si hay) ---
if no_encontrados:
    with open(ARCHIVO_NO_ENCONTRADOS, "w", encoding="utf-8") as f:
        for nombre in no_encontrados:
            f.write(nombre + "\n")
    print(f"⚠️ Juegos no encontrados: {len(no_encontrados)} (ver {ARCHIVO_NO_ENCONTRADOS})")
else:
    print("✅ Todos los juegos fueron encontrados correctamente.")


🔍 (1/91) Buscando: Fortnite


NameError: name 'res' is not defined

# Integrtacion con RAWG funcionando correctamente

In [ ]:
import pandas as pd
import requests
import time
import os

# --- CONFIGURACIÓN ---
API_KEY = "ddd05c91688a43bbad088303d18cf599"  # 👈 Pega aquí tu API Key de RAWG
ARCHIVO_ENTRADA = "backloggd_juegos_con_logs.xlsx"
ARCHIVO_SALIDA = "backloggd_juegos_enriquecido.xlsx"
ARCHIVO_FALLOS = "errores_rawg.txt"
PAUSA = 2.5  # segundos
INTENTOS_MAX = 3
GUARDAR_CADA = 5

# --- Leer Excel ---
df = pd.read_excel(ARCHIVO_ENTRADA)

# --- Agregar columnas si no existen ---
nuevas_columnas = ["Nombre RAWG", "Géneros", "Desarrollador", "Plataformas", "Fecha de salida", "Metacritic"]
for col in nuevas_columnas:
    if col not in df.columns:
        df[col] = ""

# --- Reanudar desde archivo existente si ya hay uno ---
if os.path.exists(ARCHIVO_SALIDA):
    df_prev = pd.read_excel(ARCHIVO_SALIDA)
    for i in range(len(df_prev)):
        for col in nuevas_columnas:
            df.at[i, col] = df_prev.at[i, col]

# --- Búsqueda en RAWG ---
def buscar_rawg(nombre, reintentos=INTENTOS_MAX):
    for intento in range(reintentos):
        try:
            url_busqueda = "https://api.rawg.io/api/games"
            params = {"search": nombre, "key": API_KEY, "page_size": 1}
            res = requests.get(url_busqueda, params=params)
            if res.status_code != 200:
                time.sleep(3)
                continue

            data = res.json()
            if not data.get("results"):
                return None

            juego = data["results"][0]
            slug = juego.get("slug", "")
            url_detalle = f"https://api.rawg.io/api/games/{slug}?key={API_KEY}"
            detalle = requests.get(url_detalle).json()

            return {
                "Nombre RAWG": juego.get("name", ""),
                "Géneros": ", ".join([g["name"] for g in juego.get("genres", [])]),
                "Desarrollador": ", ".join([d["name"] for d in detalle.get("developers", [])]) if "developers" in detalle else "",
                "Plataformas": ", ".join([p["platform"]["name"] for p in juego.get("platforms", [])]),
                "Fecha de salida": juego.get("released", ""),
                "Metacritic": juego.get("metacritic", "")
            }
        except Exception as e:
            print(f"⚠️ Error en intento {intento+1} para '{nombre}': {e}")
            time.sleep(3)
    return None

# --- Proceso principal ---
errores = []

for index, row in df.iterrows():
    #if pd.notna(row["Nombre RAWG"]):  # Ya procesado
     #   continue

    nombre = row["Juego"]
    print(f"🔍 ({index+1}/{len(df)}) Buscando: {nombre}")

    datos = buscar_rawg(nombre)

    if datos:
        for key, value in datos.items():
            df.at[index, key] = value
    else:
        print(f"❌ No encontrado: {nombre}")
        errores.append(nombre)

    # Guardado incremental
    if (index + 1) % GUARDAR_CADA == 0 or index == len(df) - 1:
        df.to_excel(ARCHIVO_SALIDA, index=False)
        print(f"💾 Guardado intermedio en {ARCHIVO_SALIDA}")

    time.sleep(PAUSA)

# --- Guardar errores ---
if errores:
    with open(ARCHIVO_FALLOS, "w", encoding="utf-8") as f:
        for nombre in errores:
            f.write(nombre + "\n")
    print(f"\n⚠️ Juegos no encontrados: {len(errores)} → ver {ARCHIVO_FALLOS}")
else:
    print("\n✅ Todos los juegos fueron enriquecidos correctamente.")

print(f"🎉 Proceso finalizado. Archivo completo: {ARCHIVO_SALIDA}")


🔍 (1/91) Buscando: Fortnite
🔍 (2/91) Buscando: Destiny
🔍 (3/91) Buscando: Marvel's Spider-Man: Miles Morales
